# Interacting Maps — live demo

Animated view of the Interacting Maps network (Cook et al., IJCNN 2011, or the
energy-based Martel 2019 thesis network). Sole input: `V` (temporal
derivative). `I`, `G`, `F`, `R` are all inferred.

Notebook counterpart of `demo.py`. Each cell is one atomic step of the
pipeline; all computation is imported from the core modules
(`config`, `interacting_maps.*`) — this notebook only wires them together.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec
from IPython.display import HTML

from config import DATASET_CONFIGS, THESIS_PARAMS, COOK_PARAMS, ITERS_PER_FRAME, \
    get_dataset_paths
from interacting_maps.data_loader import EventFrameSequence
from interacting_maps.eval_io import load_imu, get_gyro_for_frame
from interacting_maps.network import InteractingMaps
from interacting_maps.network_dissertation import InteractingMapsThesis
from interacting_maps.viz import normalise, flow_to_rgb

## Configuration

In [ ]:
DATASET = 'poster_rotation'
USE_THESIS_VERSION = True
USE_IMU = True  # only applies when USE_THESIS_VERSION = True

# Distortion handling: 'undistort_events' | 'C_full'
DIST_MODE = 'C_full'

## Resolve dataset config, paths, and the warm-start rotation

In [ ]:
cfg = DATASET_CONFIGS[DATASET]
paths = get_dataset_paths(DATASET)

initial_R = cfg['initial_R']
if initial_R is None:
    t0, dt = cfg['t_start'], cfg['frame_duration']
    initial_R = get_gyro_for_frame(load_imu(paths['imu']), t0, t0 + dt) * dt
    print(f"Auto-initialized R from IMU: {initial_R}")

T_START = cfg['t_start']
FRAME_DURATION = cfg['frame_duration']
N_FRAMES = cfg['n_frames']

## Load the event-frame sequence

In [ ]:
seq = EventFrameSequence(
    paths['events'], paths['calib'],
    frame_duration=cfg['frame_duration'],
    t_start=cfg['t_start'],
    n_frames=cfg['n_frames'],
    clip_value=10.0,
    sensor_size=cfg.get('sensor_size', None),
    undistort=(DIST_MODE == 'undistort_events'),        
)
frames = list(seq)
calib, H, W = seq.calib, seq.H, seq.W
fx, fy, cx, cy = calib.fx, calib.fy, calib.cx, calib.cy
print(f"Loaded {len(frames)} frames from {DATASET} ({H}x{W})")

## Resolve distortion handling for the kinematic matrix `C`

In [ ]:
if DIST_MODE == 'undistort_events':
    dist_coeffs = None       # events already undistorted -> pinhole C
elif DIST_MODE == 'C_full':
    dist_coeffs = calib.dist  # distortion-aware C (+ Jacobian)
else:
    raise ValueError(f"Unknown DIST_MODE: {DIST_MODE}")

## Build and initialize the network

In [ ]:
if USE_THESIS_VERSION:
    net_params = THESIS_PARAMS.copy()
    net = InteractingMapsThesis(H=H, W=W, fx=fx, fy=fy, cx=cx, cy=cy,
                                dist_coeffs=dist_coeffs, **net_params)
    net.initialize_from_rotation(initial_R)
else:
    net_params = COOK_PARAMS.copy()
    net = InteractingMaps(H=H, W=W, fx=fx, fy=fy, cx=cx, cy=cy,
                          dist_coeffs=dist_coeffs, **net_params)
    net.reset(scale=0.5)
    net.R = initial_R.copy()
    net.F = np.einsum('hwij,j->hwi', net._C_mat, initial_R)  # seed OFCE

## Load IMU gyro (thesis + IMU anchor only)

In [ ]:
imu_data = load_imu(paths['imu']) if (USE_THESIS_VERSION and USE_IMU) else None

## Figure setup

In [ ]:
fig = plt.figure(figsize=(14, 8))
fig.suptitle(
    f'Interacting Maps — {"Thesis (Martel 2019)" if USE_THESIS_VERSION else "Cook et al. 2011"}  '
    f'[{DATASET}]\n'
    'Sole input: V (temporal derivative). All other maps are inferred.',
    fontsize=10,
)

gs = GridSpec(2, 4, figure=fig, hspace=0.40, wspace=0.30)
ax_V = fig.add_subplot(gs[0, 0])
ax_I = fig.add_subplot(gs[0, 1])
ax_G = fig.add_subplot(gs[0, 2])
ax_F = fig.add_subplot(gs[0, 3])
ax_R = fig.add_subplot(gs[1, 0])
ax_res = fig.add_subplot(gs[1, 1:])

blank, blank3 = np.zeros((H, W)), np.zeros((H, W, 3))
im_V = ax_V.imshow(blank, cmap='RdBu', vmin=-1, vmax=1)
im_I = ax_I.imshow(blank, cmap='gray', vmin=0, vmax=1)
im_G = ax_G.imshow(blank, cmap='hot', vmin=0, vmax=1)
im_F = ax_F.imshow(blank3)

for ax, t in [(ax_V, 'V: Input (temporal deriv.)'),
              (ax_I, 'I: Inferred intensity'),
              (ax_G, '|G|: Inferred gradient'),
              (ax_F, 'F: Inferred flow (HSV)')]:
    ax.set_title(t, fontsize=8)
    ax.axis('off')

r_idx = np.arange(3)
bars_R = ax_R.bar(r_idx, [0, 0, 0], 0.6, color=['#4e79a7', '#f28e2b', '#e15759'])
ax_R.set_xticks(r_idx)
ax_R.set_xticklabels(['Rx', 'Ry', 'Rz'])
ax_R.set_title('Estimated camera rotation R (rad/frame)', fontsize=8)
ax_R.axhline(0, color='k', linewidth=0.5)
ax_R.set_ylim(-0.05, 0.05)

res_VFG_hist, res_GI_hist = [], []
line_VFG, = ax_res.semilogy([], [], 'b-', lw=1.5, label='|V + F·G|  (flow constraint)')
line_GI, = ax_res.semilogy([], [], 'r-', lw=1.5, label='|G − ∇I|   (gradient constraint)')
ax_res.set_title('Constraint residuals over frames', fontsize=8)
ax_res.set_xlabel('Frame')
ax_res.legend(fontsize=7)
ax_res.set_xlim(0, N_FRAMES)
ax_res.set_ylim(1e-5, 2)

plt.close(fig)  # don't show the static scaffold; the animation renders it below

## Per-frame update (one step of the network + one redraw)

In [ ]:
def update(t):
    if t >= len(frames):
        return im_V, im_I, im_G, im_F, *bars_R, line_VFG, line_GI

    V, _ = frames[t]

    if USE_THESIS_VERSION and USE_IMU:
        t_lo = T_START + t * FRAME_DURATION
        t_hi = t_lo + FRAME_DURATION
        omega_imu = get_gyro_for_frame(imu_data, t_lo, t_hi)
        net.step(V, n_iters=ITERS_PER_FRAME, omega_imu=omega_imu)
    else:
        net.step(V, n_iters=ITERS_PER_FRAME)

    im_V.set_data(V)
    vabs = max(np.percentile(np.abs(V), 95), 1e-3)
    im_V.set_clim(-vabs, vabs)

    im_I.set_data(normalise(net.I[:H, :W]))
    im_G.set_data(normalise(np.linalg.norm(net.G, axis=-1)))
    im_F.set_data(flow_to_rgb(net.F))

    for bar, val in zip(bars_R, net.R):
        bar.set_height(val)
    ax_R.set_ylim(min(-0.05, net.R.min() * 1.5), max(0.05, net.R.max() * 1.5))

    res_VFG_hist.append(net.residual_VFG(V) + 1e-9)
    res_GI_hist.append(net.residual_GI() + 1e-9)
    xs = list(range(len(res_VFG_hist)))
    line_VFG.set_data(xs, res_VFG_hist)
    line_GI.set_data(xs, res_GI_hist)

    sparsity = np.mean(V == 0) * 100
    print(f'Frame {t+1:3d}/{N_FRAMES}  res_VFG={res_VFG_hist[-1]:.4f}  '
          f'res_GI={res_GI_hist[-1]:.5f}  V_sparsity={sparsity:.0f}%  '
          f'|R|={np.linalg.norm(net.R):.3e}')

    return im_V, im_I, im_G, im_F, *bars_R, line_VFG, line_GI

## Run the animation

In [ ]:
ani = animation.FuncAnimation(
    fig, update, frames=N_FRAMES, interval=100, blit=False, repeat=False,
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
HTML(ani.to_jshtml())

## Final estimated rotation

In [ ]:
print('Final estimated rotation R:')
print(f'  R = {net.R}')
print(f'  |R| = {np.linalg.norm(net.R):.4e}')
print(f'  ω = R/dt = {net.R / FRAME_DURATION} rad/s')